In [87]:
import pandas as pd
import numpy as np
import joblib
from keras.models import load_model
import ast

In [88]:
all_emails = pd.read_csv('../data/04_all_emails_with_labelled_samples.csv')

## Mask all embeddings that are not labelled

In [89]:
# check type of DISC_final
print(all_emails['DISC_final'].apply(lambda x: type(x)).value_counts())

DISC_final
<class 'str'>    64442
Name: count, dtype: int64


In [90]:
# convert DISC_final to list
all_emails['DISC_final'] = all_emails['DISC_final'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else [])
print(all_emails['DISC_final'].value_counts())

DISC_final
[]     62054
[S]      674
[C]      629
[I]      582
[D]      503
Name: count, dtype: int64


In [91]:
not_labelled_mask = all_emails['DISC_final'].apply(lambda x: len(x) == 0)
print(all_emails[not_labelled_mask]['DISC_final'].value_counts())


DISC_final
[]    62054
Name: count, dtype: int64


In [92]:
X = np.load('../model_data/X_all_bert.npy')
print(X)

[[-0.7299717  -0.49167195 -0.8787571  ... -0.8831387  -0.66493493
   0.798456  ]
 [-0.62676775 -0.42787698 -0.92575437 ... -0.74408275 -0.58848137
   0.7045142 ]
 [-0.61534685 -0.40817738 -0.69530636 ... -0.2173199  -0.6889094
   0.74455506]
 ...
 [-0.5463044  -0.50168735 -0.97612244 ... -0.85623616 -0.59687084
   0.34431025]
 [-0.74502814 -0.53911716 -0.97416407 ... -0.92620146 -0.6500866
   0.65626997]
 [-0.7310124  -0.5802631  -0.9622662  ... -0.8861105  -0.66297936
   0.69729215]]


In [93]:
X_not_labelled = X[not_labelled_mask]

## Predict DISC Labels with Logistic Regression

In [104]:
log_reg = joblib.load('../models/log_reg_bootstrap.pkl')

In [95]:
y_pred_probi_log_reg = log_reg.predict_proba(X_not_labelled)
print(y_pred_probi_log_reg)

threshold_log_reg = 0.2
y_pred_log_reg = (y_pred_probi_log_reg >= threshold_log_reg).astype(int)

print(y_pred_log_reg)

[[0.22537586 0.32320042 0.14260739 0.13101938]
 [0.14248023 0.09783818 0.25768252 0.5436792 ]
 [0.25281834 0.10374325 0.26084311 0.17657075]
 ...
 [0.15141419 0.05027271 0.1471198  0.75255273]
 [0.09834303 0.0935619  0.49034689 0.30100052]
 [0.05465415 0.23861551 0.16082456 0.64775935]]
[[1 1 0 0]
 [0 0 1 1]
 [1 0 1 0]
 ...
 [0 0 0 1]
 [0 0 1 1]
 [0 1 0 1]]


In [96]:
disc_probi = ['']

In [97]:
disc_labels = ['D', 'I', 'S', 'C']
log_reg_df = pd.DataFrame(y_pred_log_reg, columns=disc_labels)

In [98]:
log_reg_df['DISC'] = log_reg_df.apply(lambda x: [disc_labels[i] for i in range(4) if x[disc_labels[i]] == 1], axis=1)

In [103]:
log_reg_df['msg_embeddings'] = X_not_labelled.tolist()
log_reg_df.sample(10)

,D,I,S,C,DISC,msg_embeddings
15730,1,0,0,0,[D],"[-0.5581594109535217, -0.37679678201675415, -0..."
43987,0,1,0,1,"[I, C]","[-0.7501011490821838, -0.4709327518939972, -0...."
47716,0,1,1,0,"[I, S]","[-0.3271554708480835, -0.4237968325614929, -0...."
23016,1,0,0,1,"[D, C]","[-0.6185435056686401, -0.5179823637008667, -0...."
31808,0,0,0,1,[C],"[-0.7144622206687927, -0.637109637260437, -0.9..."
46995,0,0,1,1,"[S, C]","[-0.5200128555297852, -0.5189606547355652, -0...."
10582,1,1,0,0,"[D, I]","[-0.6533252000808716, -0.49984216690063477, -0..."
19750,1,1,1,0,"[D, I, S]","[-0.3792301118373871, -0.5263845920562744, -0...."
19787,0,1,1,1,"[I, S, C]","[-0.7122155427932739, -0.49924615025520325, -0..."
13349,0,1,1,0,"[I, S]","[-0.047351378947496414, -0.005928334314376116,..."


In [105]:
log_reg_df.to_csv('../data/07_predicted_labels_all_emails.csv', index=False)